# From a single neuron to ResNet — build a CIFAR-10 image classifier

_Capstones_

**Walk the path from the backprop rule to a small ResNet, one landmark paper at a time, then assemble and train the working classifier.**

---

This notebook is a practice scaffold for the **From a single neuron to ResNet — build a CIFAR-10 image classifier** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the CIFAR10 dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
print("dataset: CIFAR10   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torchvision, torchvision.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1. The residual block: conv -> BN -> ReLU -> conv -> BN, then + x, then ReLU. ---
#     skip=True  -> ResNet (the residual block, ResNet 2015).
#     skip=False -> the ABLATION: a matched plain net (same depth, no '+ identity').
#     Pieces by paper: nn.Conv2d (LeNet, 3x3 per VGG), nn.BatchNorm2d (BatchNorm),
#                      nn.ReLU (AlexNet), out + identity (ResNet).
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, skip=True):
        super().__init__()
        self.skip  = skip
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        # Projection shortcut (ResNet Eqn. 2, W_s): only when channels or stride change.
        self.proj = None
        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))   # conv -> BN -> ReLU
        out = self.bn2(self.conv2(out))            # conv -> BN  (this is F(x))
        if self.skip:                              # the ablation switch
            if self.proj is not None:
                identity = self.proj(x)            # W_s x  when dims differ
            out = out + identity                   # ResNet Eqn. 1: F(x) + x
        return self.relu(out)                      # ReLU AFTER the add


# --- 2. Stack blocks into stages -> the small ResNet (or plain net if skip=False). ---
class SmallResNet(nn.Module):
    def __init__(self, blocks_per_stage=2, skip=True, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        def stage(in_ch, out_ch, stride):
            layers = [BasicBlock(in_ch, out_ch, stride, skip)]
            layers += [BasicBlock(out_ch, out_ch, 1, skip) for _ in range(blocks_per_stage - 1)]
            return nn.Sequential(*layers)
        self.stage1 = stage(16, 16, 1)
        self.stage2 = stage(16, 32, 2)             # downsample + double channels -> projection
        self.stage3 = stage(32, 64, 2)
        self.head   = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage3(self.stage2(self.stage1(x)))
        x = x.mean(dim=(2, 3))                      # global average pool
        return self.head(x)


# --- 3. CIFAR-10 subsets (torchvision is preinstalled in Colab -- no pip). ---
tfm = T.Compose([T.ToTensor(),
                 T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])
train_full = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=tfm)
test_full  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=tfm)
train_set  = torch.utils.data.Subset(train_full, range(8000))   # small + fast
test_set   = torch.utils.data.Subset(test_full,  range(2000))   # held-out images
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256)


def accuracy(net):
    net.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (net(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)
    return 100.0 * correct / total


def train(skip, epochs=8):
    torch.manual_seed(0)
    net = SmallResNet(blocks_per_stage=2, skip=skip).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)   # Adam (2014)
    lf  = nn.CrossEntropyLoss()
    for ep in range(epochs):
        net.train(); tot = nb = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = lf(net(xb), yb)
            loss.backward()                              # backprop (1986)
            opt.step()
            tot += loss.item(); nb += 1
        print(f"  epoch {ep}  train loss {tot/nb:.4f}  test acc {accuracy(net):.1f}%")
    return accuracy(net)


# --- 4. Train the assembled ResNet and PRINT test accuracy. ---
print("ResNet (skip=True) -- the assembled classifier:")
res_acc = train(skip=True)

# --- 5. The skip-removal ABLATION: same depth, no '+ identity'. ---
print("Plain net (skip=False) -- ABLATION, same depth, no residual:")
plain_acc = train(skip=False)

print(f"\nFINAL  ResNet test accuracy: {res_acc:.1f}%")
print(f"FINAL  Plain  test accuracy: {plain_acc:.1f}%")
print(f"Skip connection bought us +{res_acc - plain_acc:.1f} points of test accuracy.")
# The residual net trains faster and scores higher than the matched plain net.
# (Exact numbers vary by hardware/seed; this is our small run, not any paper's reported number.)

## Visualize the data & results

_Does our assembled small ResNet train (accuracy up, loss down), and does removing the skip hurt?_

In [ ]:
# Reproduce the two curves above: train the assembled small ResNet and the matched
# plain net (skip removed) on a CIFAR-10 subset, recording test accuracy + train loss
# per epoch. This is the same network as the CODE cell above; we just log the history.
#
# Run the CODE cell first (it defines BasicBlock, SmallResNet, the loaders, accuracy()).
import torch, torch.nn as nn

def run(skip, epochs=8):
    torch.manual_seed(0)
    net = SmallResNet(blocks_per_stage=2, skip=skip).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    lf  = nn.CrossEntropyLoss()
    accs, losses = [], []
    for ep in range(epochs):
        net.train(); tot = nb = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = lf(net(xb), yb); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        losses.append(round(tot / nb, 3))
        accs.append(round(accuracy(net), 1))
    return accs, losses

res_acc,  res_loss  = run(skip=True)
plain_acc, plain_loss = run(skip=False)
print("ResNet test acc :", res_acc)
print("Plain  test acc :", plain_acc)
print("ResNet train loss:", res_loss)
print("Plain  train loss:", plain_loss)
# ResNet climbs higher and its loss falls faster than the matched plain net.
# (Our small run, not any paper's reported number; exact values vary by hardware/seed.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The capstone ablation. You have just trained the assembled small ResNet and printed its
            CIFAR-10 test accuracy. Now set skip=False to delete the single
            "+ identity" line in every block (a matched plain net of identical depth, width,
            optimizer, and data) and retrain. What happens to test accuracy, and what does the
            comparison prove about which paper's idea is load-bearing?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: the block's skip flag from True to False, so out = relu(out + identity) becomes out = relu(out). Keep depth, width, BatchNorm, Adam, seed, and the CIFAR-10 subset identical. — _An honest ablation varies one factor &mdash; the residual connection &mdash; so any accuracy gap is attributable to it alone, not to capacity or tuning._
- Retrain and compare: the residual net's test accuracy is clearly higher; the deep plain net trains more slowly and lands lower (or stalls). — _Without the "$+x$" gradient highway, the signal to early layers is weak, so the deep plain stack optimizes poorly &mdash; the degradation problem reproduced on our small run._
- Conclude that the skip connection, not extra parameters, is what makes the deep classifier work. — _Both nets share layers and parameter count; only the residual one trains well, isolating ResNet's contribution from all the others._

**Answer:** The residual net reaches clearly higher CIFAR-10 test accuracy than the matched
                 plain net, which trains slowly and plateaus low. Because the two are identical
                 except for the "+ identity", this isolates ResNet's skip connection as
                 the load-bearing idea for depth &mdash; an optimization fix, not a parameter-count
                 or regularization effect. The CODEVIZ panel shows the same gap in the training curves.
                 (These are our small-run numbers, not any paper's reported result.)

</details>

**Problem 2.** Trace one component to one line of the final build. For each of (a) convolution, (b) Batch
            Normalization, (c) the residual skip, (d) the optimizer, name the paper that introduced it and
            the PyTorch construct in our code that embodies it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Map convolution &rarr; LeNet &rarr; nn.Conv2d(...) (the 3&times;3 layers, with VGG's stacking recipe). — _LeNet introduced weight-shared sliding filters; VGG fixed the size at 3&times;3 and stacked them._
- Map normalization &rarr; BatchNorm &rarr; nn.BatchNorm2d(...) after each conv. — _Standardize-then-scale per channel over the mini-batch is exactly the BatchNorm paper._
- Map the skip &rarr; ResNet &rarr; out = relu(out + identity); and the step rule &rarr; Adam &rarr; torch.optim.Adam(...). ReLU traces to AlexNet, and loss.backward() to backprop. — _Each line of the build is one paper's contribution made concrete._

**Answer:** (a) Convolution &rarr; LeNet &rarr; nn.Conv2d (3&times;3, stacked per
                 VGG). (b) Batch Normalization &rarr; BatchNorm &rarr; nn.BatchNorm2d.
                 (c) The residual skip &rarr; ResNet &rarr; out = relu(out + identity).
                 (d) The optimizer &rarr; Adam &rarr; torch.optim.Adam. (ReLU is
                 AlexNet's; the backward sweep loss.backward() is backprop.) The
                 whole network is these seven papers assembled.

</details>